# DAiSEE CNN-LSTM: end-to-end guided notebook
Run cells in order. First use a small sample to validate paths, labels, and runtime. This notebook predicts engagement intensity labels only; fatigue needs separate ground-truth data.

## 1. Install and import packages

In [ ]:
!pip -q install opencv-python-headless scikit-learn seaborn
import os, random, time
from pathlib import Path
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
random.seed(42); np.random.seed(42); tf.random.set_seed(42)
print(tf.__version__, tf.config.list_physical_devices("GPU"))

## 2. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/learner_engagement_project')
VIDEO_ROOT = PROJECT_DIR/'data/raw/DAiSEE'       # CHANGE to the directory containing extracted videos
META_DIR = PROJECT_DIR/'data/metadata'           # CHANGE if needed
ARTIFACT_DIR = PROJECT_DIR/'artifacts'
MODEL_DIR = PROJECT_DIR/'models'
for p in [ARTIFACT_DIR, MODEL_DIR]: p.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = META_DIR/'TrainLabels.csv'           # CHANGE to actual filename
VAL_CSV = META_DIR/'ValidationLabels.csv'        # CHANGE to actual filename
TEST_CSV = META_DIR/'TestLabels.csv'             # CHANGE to actual filename
IMG_SIZE=(224,224); NUM_FRAMES=16; BATCH_SIZE=8
MAX_CLIPS=100  # smoke test; change to None only after confirming pipeline and resources

## 3. Load metadata and inspect columns

In [ ]:
train_df=pd.read_csv(TRAIN_CSV); val_df=pd.read_csv(VAL_CSV); test_df=pd.read_csv(TEST_CSV)
for name,df in [('train',train_df),('validation',val_df),('test',test_df)]:
    print(name, df.shape, df.columns.tolist())
    display(df.head())
# Set these to exact column names printed above:
CLIP_ID_COL='ClipID'
LABEL_COL='Engagement'

## 4. Verify labels and prepare target IDs

In [ ]:
for name,df in [('train',train_df),('val',val_df),('test',test_df)]:
    print(name, sorted(df[LABEL_COL].dropna().unique().tolist()), "missing:", df[LABEL_COL].isna().sum())
# Common DAiSEE intensity encoding is 0..3; VERIFY against official metadata/docs.
LABEL_TO_ID={0:0,1:1,2:2,3:3}
LABEL_NAMES=['Very Low','Low','High','Very High']
def prep(df):
    x=df.copy()
    x['label_id']=x[LABEL_COL].map(LABEL_TO_ID)
    if x['label_id'].isna().any():
        raise ValueError("Unmapped label values: "+str(x.loc[x.label_id.isna(),LABEL_COL].unique()))
    x['label_id']=x['label_id'].astype('int32')
    return x
train_df, val_df, test_df=map(prep,[train_df,val_df,test_df])

## 5. Index video files and match metadata

In [ ]:
VIDEO_EXTS={'.avi','.mp4','.mov','.mkv'}
video_index={}
for p in Path(VIDEO_ROOT).rglob('*'):
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
        video_index.setdefault(p.stem,p)
def attach_paths(df):
    x=df.copy()
    x['video_path']=x[CLIP_ID_COL].astype(str).map(video_index)
    print("unmatched videos:",x.video_path.isna().sum(),"of",len(x))
    return x.dropna(subset=['video_path']).reset_index(drop=True)
train_df,val_df,test_df=map(attach_paths,[train_df,val_df,test_df])
# Keep the official train/validation/test split. Check participant IDs for overlap if available.

## 6. EDA: label counts and data quality

In [ ]:
for name,df in [('Train',train_df),('Validation',val_df),('Test',test_df)]:
    print(name, len(df)); display(df.label_id.value_counts().sort_index())
print("Missing values:\\n",train_df.isna().sum())
plt.figure(figsize=(8,4))
sns.countplot(data=train_df,x='label_id',order=[0,1,2,3])
plt.xticks([0,1,2,3],LABEL_NAMES); plt.title('Training engagement label distribution')
plt.tight_layout(); plt.savefig(ARTIFACT_DIR/'label_distribution.png',dpi=150); plt.show()

## 7. Sample frames and inspect a clip

In [ ]:
def sample_video(path,num_frames=NUM_FRAMES,image_size=IMG_SIZE):
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened(): raise ValueError(f"Cannot open {path}")
    n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n<=0: cap.release(); raise ValueError("No frames")
    indices=np.linspace(0,n-1,num_frames).astype(int); frames=[]
    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(i)); ok,frame=cap.read()
        if not ok: cap.release(); raise ValueError(f"Cannot read frame {i}")
        frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        frame=cv2.resize(frame,image_size)
        frames.append(frame.astype('float32'))
    cap.release()
    return np.asarray(frames)
frames=sample_video(train_df.iloc[0].video_path)
fig,axes=plt.subplots(2,4,figsize=(12,6))
for ax,im in zip(axes.flat,frames[:8]):
    ax.imshow(im.astype('uint8')); ax.axis('off')
plt.show()

## 8. Build pretrained MobileNetV2 feature extractor

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
backbone=MobileNetV2(input_shape=(224,224,3),include_top=False,weights='imagenet')
backbone.trainable=False
inp=tf.keras.Input((224,224,3))
x=preprocess_input(inp); x=backbone(x,training=False)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
encoder=tf.keras.Model(inp,x,name='frame_encoder')
encoder.summary()

## 9. Extract frame features for each clip

In [ ]:
def clip_features(path):
    fr=sample_video(path)
    # Encoder already applies MobileNetV2 preprocess_input internally.
    return encoder.predict(fr,batch_size=NUM_FRAMES,verbose=0).astype('float32')
def make_arrays(df,name):
    part=df if MAX_CLIPS is None else df.head(MAX_CLIPS)
    X=[]; y=[]; failures=[]
    for row in part.itertuples(index=False):
        try:
            X.append(clip_features(row.video_path)); y.append(int(row.label_id))
        except Exception as e:
            failures.append((str(row.video_path),str(e)))
    print(name,'success',len(X),'failures',len(failures),failures[:3])
    return np.asarray(X,dtype='float32'),np.asarray(y,dtype='int32')
X_train,y_train=make_arrays(train_df,'train')
X_val,y_val=make_arrays(val_df,'validation')
X_test,y_test=make_arrays(test_df,'test')
print(X_train.shape,y_train.shape)
np.savez_compressed(ARTIFACT_DIR/'features.npz',X_train=X_train,y_train=y_train,
                     X_val=X_val,y_val=y_val,X_test=X_test,y_test=y_test)

## 10. Train the temporal LSTM classifier

In [ ]:
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(NUM_FRAMES,X_train.shape[-1])),
    tf.keras.layers.LSTM(128,dropout=0.3),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(4,activation='softmax')
],name='cnn_lstm_engagement')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',metrics=['accuracy'])
classes=np.unique(y_train)
weights=compute_class_weight(class_weight='balanced',classes=classes,y=y_train)
class_weights={int(c):float(w) for c,w in zip(classes,weights)}
callbacks=[
 tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True),
 tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=2),
 tf.keras.callbacks.ModelCheckpoint(str(MODEL_DIR/'best_cnn_lstm.keras'),monitor='val_loss',save_best_only=True)
]
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=30,
                  batch_size=BATCH_SIZE,class_weight=class_weights,callbacks=callbacks)

## 11. Plot training curves

In [ ]:
hist=pd.DataFrame(history.history)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(hist.loss,label='train'); ax[0].plot(hist.val_loss,label='validation'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(hist.accuracy,label='train'); ax[1].plot(hist.val_accuracy,label='validation'); ax[1].set_title('Accuracy'); ax[1].legend()
plt.tight_layout(); plt.savefig(ARTIFACT_DIR/'training_curves.png',dpi=150); plt.show()

## 12. Evaluate on validation data and inspect errors

In [ ]:
val_pred=model.predict(X_val,verbose=0).argmax(axis=1)
print(classification_report(y_val,val_pred,labels=[0,1,2,3],target_names=LABEL_NAMES,zero_division=0))
cm=confusion_matrix(y_val,val_pred,labels=[0,1,2,3])
sns.heatmap(cm,annot=True,fmt='d',xticklabels=LABEL_NAMES,yticklabels=LABEL_NAMES,cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout()
plt.savefig(ARTIFACT_DIR/'validation_confusion_matrix.png',dpi=150); plt.show()

## 13. Final test evaluation (only after model selection)

In [ ]:
test_pred=model.predict(X_test,verbose=0).argmax(axis=1)
print(classification_report(y_test,test_pred,labels=[0,1,2,3],target_names=LABEL_NAMES,zero_division=0))
print('Accuracy:',accuracy_score(y_test,test_pred))
print('Macro F1:',f1_score(y_test,test_pred,average='macro',zero_division=0))
print('Balanced accuracy:',balanced_accuracy_score(y_test,test_pred))

## 14. Save models for the dashboard

In [ ]:
model.save(MODEL_DIR/'cnn_lstm.keras')
encoder.save(MODEL_DIR/'frame_encoder.keras')
print('Saved model files:',list(MODEL_DIR.glob('*.keras')))

## 15. Dashboard and GitHub

In [ ]:
# In a local terminal, from the project root:
# pip install -r requirements.txt
# streamlit run app.py
#
# GitHub (create a PRIVATE empty repo first):
# git init
# git add .
# git commit -m "Initial learner engagement project"
# git branch -M main
# git remote add origin https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git
# git push -u origin main
#
# Never push data/, credentials, or model weights unless you have reviewed size/licensing.